[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.4_disaggregated_serving/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.4_disaggregated_serving/lab.ipynb)

# Lab 7.4: Disaggregated Serving

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.4_disaggregated_serving/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/08_serving/07.4_disaggregated_serving/lab.ipynb)

Explore disaggregated prefill/decode architecture: KV transfer costs, pool sizing,
pipelined scheduling, and throughput comparison vs aggregated serving.

In [ ]:
# Install dependencies using subprocess (works on Colab and Molab)
import subprocess, sys
# Install numpy for numerical computation and matplotlib for visualization
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
# Import numpy for array operations and math
import numpy as np
# Import matplotlib for plotting charts
import matplotlib.pyplot as plt

# Set random seed for reproducible simulation results
np.random.seed(42)
# Use a clean grid-based plot style
plt.style.use('seaborn-v0_8-whitegrid')

## 1. KV Cache Size Per Request

The KV cache that must transfer between prefill and decode pools grows linearly
with sequence length. This cell computes the size for a Mistral-7B-style model.

In [ ]:
# === MODEL PARAMETERS (Mistral-7B architecture) ===
# Number of transformer layers in the model
NUM_LAYERS = 32
# Number of KV heads (GQA: Mistral uses 8 KV heads shared across 32 query heads)
NUM_KV_HEADS = 8
# Dimension of each attention head
HEAD_DIM = 128
# Bytes per element (FP16 = 2 bytes)
DTYPE_BYTES = 2

def kv_cache_size_bytes(seq_len):
    """Compute KV cache size in bytes for one request.
    Formula: 2 (K+V) * layers * kv_heads * head_dim * seq_len * dtype_bytes.
    The factor of 2 accounts for both Key and Value tensors."""
    # Multiply all dimensions together to get total bytes
    return 2 * NUM_LAYERS * NUM_KV_HEADS * HEAD_DIM * seq_len * DTYPE_BYTES

# Define sequence lengths to evaluate (from short to very long)
seq_lens = [512, 1024, 2048, 4096, 8192, 16384, 32768]
# Convert bytes to megabytes for readability
sizes_mb = [kv_cache_size_bytes(s) / 1e6 for s in seq_lens]

# Create bar chart showing KV cache size growth
fig_2, ax_2 = plt.subplots(figsize=(8, 4))
# Plot bars with blue color and black borders
ax_2.bar([str(s) for s in seq_lens], sizes_mb, color='#2563eb', edgecolor='black')
# Label axes_2 to explain what we're measuring
ax_2.set_xlabel('Sequence Length (tokens)')
ax_2.set_ylabel('KV Cache Size (MB)')
ax_2.set_title('Per-Request KV Cache Size (32L, 8 KV heads, d=128, FP16)')
plt.tight_layout()
plt.show()
# Print the largest case to show the scale of the transfer problem
print(f'32K sequence -> {sizes_mb[-1]:.1f} MB per request to transfer between pools')

## 2. KV Transfer Latency: RDMA vs TCP vs NVLink

Transfer time directly impacts TTFT in disaggregated systems.
We compare four interconnect options at different sequence lengths.

In [ ]:
# Define interconnect bandwidths (effective throughput after protocol overhead)
# Each entry maps interconnect name to bandwidth in GB/s
INTERCONNECTS = {
    'TCP 100GbE': 10.0,         # Standard ethernet, ~80 Gbps effective
    'RDMA 200Gb IB': 22.0,      # InfiniBand RDMA, ~176 Gbps effective
    'RDMA 400Gb IB': 42.0,      # High-end InfiniBand, ~336 Gbps effective
    'NVLink intra-node': 450.0   # NVLink 4.0, only works within same server
}

def transfer_time_ms(size_bytes, bw_gbs):
    """Calculate transfer time in milliseconds.
    Divides data size by bandwidth, converts seconds to ms."""
    # bytes / (GB/s * 1e9) gives seconds, multiply by 1000 for ms
    return (size_bytes / (bw_gbs * 1e9)) * 1000

# Test with representative sequence lengths
seq_test = [1024, 4096, 16384, 32768]

# Create grouped bar chart comparing all interconnects
fig_3, ax_3 = plt.subplots(figsize=(9, 5))
# x positions for the groups of bars
x = np.arange(len(seq_test))
# Width of each individual bar
width = 0.2
# Colors: red for slow, amber/green for mid, blue for fast
colors = ['#ef4444', '#f59e0b', '#22c55e', '#2563eb']

# Plot one bar group per interconnect type
for i, (name, bw) in enumerate(INTERCONNECTS.items()):
    # Compute transfer time for each sequence length at this bandwidth
    times = [transfer_time_ms(kv_cache_size_bytes(s), bw) for s in seq_test]
    # Offset each bar group by i*width so they don't overlap
    ax_3.bar(x + i * width, times, width, label=name, color=colors[i], edgecolor='black')

# Center x-tick labels under each group
ax_3.set_xticks(x + 1.5 * width)
ax_3.set_xticklabels([f'{s//1024}K' for s in seq_test])
# Label axes_3
ax_3.set_xlabel('Sequence Length')
ax_3.set_ylabel('Transfer Time (ms)')
ax_3.set_title('KV Cache Transfer Latency by Interconnect')
ax_3.legend()
# Use log scale because NVLink is orders of magnitude faster
ax_3.set_yscale('log')
plt.tight_layout()
plt.show()
# Key result: RDMA 400Gb keeps the critical 4K case under 2ms
print(f'4K seq via RDMA 400Gb: {transfer_time_ms(kv_cache_size_bytes(4096), 42.0):.1f} ms')
# Compare with TCP which would add significant TTFT overhead
print(f'4K seq via TCP 100GbE: {transfer_time_ms(kv_cache_size_bytes(4096), 10.0):.1f} ms')

## 3. Pool Sizing Optimization

System throughput is limited by the slower pool (prefill or decode).
We sweep the prefill/decode GPU split to find the optimal ratio.

In [ ]:
# === CLUSTER PARAMETERS ===
# Total GPUs available to split between prefill and decode
TOTAL_GPUS = 16
# Prefill throughput: tokens per second one GPU can process during prefill
PREFILL_TOKENS_PER_GPU = 50000
# Decode: how many concurrent sequences one GPU can handle
DECODE_SEQS_PER_GPU = 64
# Average prompt length in tokens (determines prefill workload)
AVG_PROMPT = 2048
# Average output length in tokens (determines decode duration)
AVG_OUTPUT = 256
# Decode speed: tokens generated per second per GPU
DECODE_TOK_PER_S = 2000

def system_throughput(n_prefill):
    """Compute system throughput (req/s) for a given prefill GPU allocation.
    The system is bottlenecked by whichever pool has lower capacity."""
    # Remaining GPUs go to decode
    n_decode = TOTAL_GPUS - n_prefill
    # Prefill capacity = total tokens/s / tokens per request
    prefill_cap = (n_prefill * PREFILL_TOKENS_PER_GPU) / AVG_PROMPT
    # Decode capacity = concurrent slots / time per request
    decode_time = AVG_OUTPUT / DECODE_TOK_PER_S  # seconds per decode request
    decode_cap = (n_decode * DECODE_SEQS_PER_GPU) / decode_time
    # System throughput is the minimum of the two (bottleneck)
    return min(prefill_cap, decode_cap)

# Sweep all valid splits from 1 to 15 prefill GPUs
splits = np.arange(1, TOTAL_GPUS)
# Compute throughput at each split point
throughputs = [system_throughput(n) for n in splits]
# Find the split that maximizes throughput
best_idx = np.argmax(throughputs)

# Plot throughput curve with optimal point marked
fig_4, ax_4 = plt.subplots(figsize=(8, 4))
# Line plot with markers at each data point
ax_4.plot(splits, throughputs, 'o-', color='#2563eb', linewidth=2, markersize=6)
# Vertical line at the optimal split
ax_4.axvline(splits[best_idx], color='red', linestyle='--',
           label=f'Optimal: {splits[best_idx]}P + {TOTAL_GPUS-splits[best_idx]}D')
# Axis labels explain the tradeoff being explored
ax_4.set_xlabel('Prefill GPUs (out of 16 total)')
ax_4.set_ylabel('System Throughput (req/s)')
ax_4.set_title('Pool Sizing: Find Optimal Prefill/Decode Split')
ax_4.legend()
plt.tight_layout()
plt.show()
# Print the optimal configuration
print(f'Optimal: {splits[best_idx]} prefill + {TOTAL_GPUS-splits[best_idx]} decode = {throughputs[best_idx]:.0f} req/s')

## 4. Pipelined vs Sequential KV Transfer (Mooncake)

Mooncake predicts output length to pre-allocate decode slots, overlapping
KV transfer with decode batch formation. We simulate both approaches.

In [ ]:
# === SIMULATION PARAMETERS ===
# Number of requests to simulate
N_REQ = 500
# Average prefill time in milliseconds
AVG_PREFILL_MS = 50
# Average decode time in milliseconds
AVG_DECODE_MS = 200
# KV transfer latency over RDMA (ms)
KV_TRANSFER_MS = 5
# How often Mooncake's output-length predictor is correct
PRED_ACCURACY = 0.85

# Sample random prefill and decode times from exponential distribution
# Exponential models the variable nature of real request processing
prefill_times = np.random.exponential(AVG_PREFILL_MS, N_REQ)
decode_times = np.random.exponential(AVG_DECODE_MS, N_REQ)

# Naive sequential: must wait for transfer to complete before decode starts
# Total = prefill + transfer_wait + decode
naive_lat = prefill_times + KV_TRANSFER_MS + decode_times

# Mooncake pipelined: transfer overlaps with decode batch formation
# Correct prediction: transfer is fully hidden (0 overhead)
# Wrong prediction: must reallocate, paying 2x transfer penalty
correct = np.random.random(N_REQ) < PRED_ACCURACY
# Where prediction is correct, no transfer overhead; otherwise 2x penalty
moon_lat = prefill_times + decode_times + np.where(correct, 0, KV_TRANSFER_MS * 2)

# Create side-by-side comparison plots
fig_5, axes_5 = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: histogram comparing latency distributions
axes_5[0].hist(naive_lat, bins=30, alpha=0.7, label='Naive Sequential', color='#ef4444')
axes_5[0].hist(moon_lat, bins=30, alpha=0.7, label='Mooncake Pipelined', color='#2563eb')
axes_5[0].set_xlabel('End-to-End Latency (ms)')
axes_5[0].set_ylabel('Count')
axes_5[0].legend()
axes_5[0].set_title('Latency Distribution')

# Right panel: tail latency comparison at standard SLO percentiles
pctls = [50, 90, 95, 99]
# Compute percentile values for both approaches
naive_p = np.percentile(naive_lat, pctls)
moon_p = np.percentile(moon_lat, pctls)
x = np.arange(len(pctls))
# Side-by-side bars at each percentile
axes_5[1].bar(x - 0.15, naive_p, 0.3, label='Naive', color='#ef4444', edgecolor='black')
axes_5[1].bar(x + 0.15, moon_p, 0.3, label='Mooncake', color='#2563eb', edgecolor='black')
axes_5[1].set_xticks(x)
axes_5[1].set_xticklabels([f'P{p}' for p in pctls])
axes_5[1].set_ylabel('Latency (ms)')
axes_5[1].set_title('Tail Latency Comparison')
axes_5[1].legend()
plt.tight_layout()
plt.show()

# Compute and print the median improvement percentage
improvement = (1 - np.median(moon_lat) / np.median(naive_lat)) * 100
print(f'Median improvement: {improvement:.1f}% lower latency with Mooncake pipelining')

## 5. Aggregated vs Disaggregated Throughput Under Load

Aggregated serving suffers prefill/decode interference at high utilization.
Disaggregated maintains linear throughput until pool capacity is reached.

In [ ]:
def throughput_comparison(arrival_rates, total_gpus=16, prefill_frac=0.375):
    """Compare achieved throughput: aggregated (interference) vs disaggregated (isolated)."""
    # Aggregated system: all GPUs handle both phases
    agg_max = total_gpus * 8  # maximum req/s with no interference
    # Disaggregated: separate pools, each operating independently
    n_pf = int(total_gpus * prefill_frac)  # prefill GPU count
    n_dc = total_gpus - n_pf  # decode GPU count
    # Disaggregated max is bottlenecked by the smaller pool
    disagg_max = min(n_pf * 20, n_dc * 10)

    agg_res, disagg_res = [], []
    for rate in arrival_rates:
        # Aggregated: prefill bursts interfere with decode, causing quadratic degradation
        util = rate / agg_max  # system utilization fraction
        # Interference factor: performance degrades as utilization increases
        interference = 1.0 - 0.3 * min(util, 1.0) ** 2
        # Achieved throughput = min(demand, capacity * interference)
        agg_res.append(min(rate, agg_max * interference))
        # Disaggregated: linear until capacity, 5% transfer overhead
        disagg_res.append(min(rate, disagg_max * 0.95))
    return agg_res, disagg_res

# Sweep arrival rates from light to heavy load
rates = np.linspace(10, 200, 50)
# Run comparison simulation
agg_tp, disagg_tp = throughput_comparison(rates)

# Plot throughput curves
fig_6, ax_6 = plt.subplots(figsize=(8, 5))
# Aggregated line (red) shows degradation at high load
ax_6.plot(rates, agg_tp, 'o-', label='Aggregated', color='#ef4444', markersize=3)
# Disaggregated line (blue) stays linear until capacity
ax_6.plot(rates, disagg_tp, 's-', label='Disaggregated', color='#2563eb', markersize=3)
# Ideal line (gray dashed) shows perfect 1:1 rate = throughput
ax_6.plot(rates, rates, '--', color='gray', alpha=0.5, label='Ideal')
ax_6.set_xlabel('Arrival Rate (req/s)')
ax_6.set_ylabel('Achieved Throughput (req/s)')
ax_6.set_title('Aggregated vs Disaggregated: Throughput Under Load')
ax_6.legend()
plt.tight_layout()
plt.show()

# Find the crossover point where disaggregated beats aggregated
crossover = next((i for i in range(len(rates)) if disagg_tp[i] > agg_tp[i]), -1)
if crossover >= 0:
    print(f'Disaggregated wins above ~{rates[crossover]:.0f} req/s arrival rate')

## Key Takeaways

1. KV transfer is the critical cost: RDMA 400Gb keeps 4K transfers under 2ms
2. Optimal pool split is typically 30-40% prefill GPUs for mixed workloads
3. Pipelining (Mooncake) hides transfer latency via output-length prediction
4. Disaggregated wins at scale where prefill/decode interference dominates
5. Transfer overhead >15% erases benefits: fast interconnect is mandatory